# Промпт «слайд — законченная мысль» (EXP-027)

EXP-023: GRPO поднял награду в основном снятием штрафов, покрытие тезисов не сдвинулось; EXP-024…026: ни одна замена
покрытия не прошла правило. Здесь — не награда, а промпт: правило `prompts.SPECIFIC_RULE` («каждый содержательный
слайд — одна законченная мысль: заголовок её называет, текст раскрывает конкретно — определение, механизм, причина
и следствие, факт с числом, именем или датой; слайд, который только называет тему или перечисляет термины, не нужен»)
после правила об объёме текста. Модели правило не сообщает тезисов — оно против слайдов из общих слов, а не против
пропуска тезиса.

Исходная T-lite (ревизия как в GRPO), T 0.9 без top-k, 25 тем validation × 2 режима × 4 ответа — основной промпт
(`old`) и с правилом (`specific`), сиды одинаковые (пары ответов). Предел ответа 12 000 токенов, контекста — 25 000
(как в GRPO). Правило выбора — `docs/decisions.md`, раздел 9, решение 16.11.2026 (EXP-027), записано до прогона.
Результат — `runs/prompt_specific/<RUN>/`.

## 1. Серверы vLLM (терминал)

Обе карты свободны (`nvidia-smi`). Карта 0 — основной промпт (порт 8000), карта 1 — с правилом (порт 8001); промпт
задаётся в запросе, разделение по картам — только чтобы шли параллельно.
```bash
source ~/vllm085/bin/activate
cd ~/Diplom
for GPU in 0 1; do
VLLM_USE_V1=0 CUDA_VISIBLE_DEVICES=$GPU nohup python -m vllm.entrypoints.openai.api_server \
    --model t-tech/T-lite-it-2.1 --revision d125c970c553de58fcee3c937d5e4867d4a448d8 --served-model-name base \
    --dtype float16 --max-model-len 25600 --gpu-memory-utilization 0.85 --generation-config vllm \
    --port 800$GPU --seed 0 > ~/vllm_p27_$GPU.log 2>&1 &
done
grep -h "startup complete" ~/vllm_p27_*.log   # готовы оба — две строки; ошибки — tail ~/vllm_p27_0.log
```
Остановить оба: `pkill -f vllm.entrypoints.openai.api_server`.

In [ ]:
import json, os, random, statistics, sys, threading, time, traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from transformers import AutoTokenizer

from presentation_rl.environment import run_episode
from presentation_rl.evaluation import evaluate_folder, paired, write_csv
from presentation_rl.rewards import load_key_points
from presentation_rl.tasks import load_tasks
from presentation_rl.tools.cache import WikiCache
from presentation_rl.tools.wiki import WikiClient
from presentation_rl.tools.wiki_tools import WikiTools
from presentation_rl.training.common import LockedTokenizer, episode_prompt, mode_task, stop_ids, vllm_generator
from presentation_rl.training.vllm_client import VLLMClient

os.environ["WIKI_PROXY"] = ""  # Википедия открыта в пространстве; туннель — docs/server.md
RUN = time.strftime("%Y%m%d_%H%M")  # продолжение после сбоя — подставить папку прежнего запуска
OUT = ROOT / "runs" / "prompt_specific" / RUN
MODEL_ID, REVISION = "t-tech/T-lite-it-2.1", "d125c970c553de58fcee3c937d5e4867d4a448d8"
CONFIGS = {"old": (False, "http://127.0.0.1:8000"), "specific": (True, "http://127.0.0.1:8001")}
TEMPERATURE, SAMPLES, MAX_NEW_TOKENS, MAX_CONTEXT, WORKERS = 0.9, 4, 12000, 25000, 48
MODES = ("search", "plain")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="validation")}
points = load_key_points(ROOT / "data" / "tasks" / "key_points" / "key_points.json")
IDS = sorted(tasks)
EPISODES = [(mode, key, s) for mode in MODES for key in IDS for s in range(SAMPLES)]
cache = WikiCache(ROOT / "data" / "sources" / "wiki_cache",
                  WikiClient(contact=os.environ.get("WIKI_CONTACT", "https://github.com/doshik722")))
print(OUT, f"| тем {len(IDS)}, эпизодов на промпт {len(EPISODES)}")
for name, (_, url) in CONFIGS.items():
    client = VLLMClient(url)
    new, finished, logp = client.generate(tokenizer.encode("Столица Франции —"), 10, "base", TEMPERATURE, 0, [])
    print(f"{name}: {url} {client.models()} | {tokenizer.decode(new)!r}")

## 2. Генерация

Эпизод сохраняется сразу: `.json`, `.txt` (эпизод после промпта), `_answer.txt`. При перезапуске готовые эпизоды
пропускаются, эпизоды с ошибкой повторяются. Сид реплики — от темы, режима, номера ответа и номера реплики, у обоих
промптов один и тот же.

In [ ]:
safe = LockedTokenizer(tokenizer)
STOPS = stop_ids(tokenizer)
PROMPTS = {name: {(mode, key): episode_prompt(tokenizer, specific)(mode_task(tasks[key], mode).request)
                  for mode in MODES for key in IDS} for name, (specific, _) in CONFIGS.items()}
for mode in MODES:
    key = IDS[0]
    print(f"{mode}: промпт {len(PROMPTS['old'][(mode, key)])} → {len(PROMPTS['specific'][(mode, key)])} токенов")


def done(folder, stem):
    path = folder / f"{stem}.json"
    return path.exists() and not json.loads(path.read_text(encoding="utf-8")).get("error")


def one_episode(client, name, mode, key, sample):
    folder, stem = OUT / name, f"{mode}_{key}_{sample}"
    row, start = {"mode": mode, "id": key, "sample": sample}, time.monotonic()
    try:
        prompt_ids = PROMPTS[name][(mode, key)]
        generate = vllm_generator(client, "base", TEMPERATURE, STOPS, ("prompt_specific", mode, key, sample))
        episode = run_episode(prompt_ids, safe, generate, WikiTools(cache) if mode == "search" else None,
                              max_new_tokens=MAX_NEW_TOKENS, max_context=MAX_CONTEXT)
        (folder / f"{stem}.txt").write_text(safe.decode(episode.ids[len(prompt_ids):], skip_special_tokens=False),
                                            encoding="utf-8")
        (folder / f"{stem}_answer.txt").write_text(episode.answer, encoding="utf-8")
        row.update(finish=episode.finish, generated_tokens=episode.generated_tokens, calls=len(episode.calls),
                   truncated_outputs=episode.truncated_outputs, error="")
    except Exception:
        row.update(error=traceback.format_exc()[-1500:])
    row["seconds"] = round(time.monotonic() - start)
    (folder / f"{stem}.json").write_text(json.dumps(row, ensure_ascii=False, indent=1), encoding="utf-8")
    return row


def run_config(name):
    client = VLLMClient(CONFIGS[name][1])
    (OUT / name).mkdir(parents=True, exist_ok=True)
    jobs = [(mode, key, s) for mode, key, s in EPISODES if not done(OUT / name, f"{mode}_{key}_{s}")]
    errors, start = 0, time.monotonic()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = [pool.submit(one_episode, client, name, *job) for job in jobs]
        for k, future in enumerate(futures, 1):
            errors += bool(future.result()["error"])
            if k % 50 == 0 or k == len(futures):
                print(f"{name}: {k}/{len(futures)} за {(time.monotonic() - start) / 60:.0f} мин, ошибок {errors}", flush=True)


threads = [threading.Thread(target=run_config, args=(name,)) for name in CONFIGS]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

## 3. Оценка

После генерации остановить серверы (`pkill -f vllm.entrypoints.openai.api_server`) — bge-m3 занимает карту 1.
Кроме метрик `evaluate_folder` — диагностика (не для выбора): доля содержательных слайдов (без титульного
и заключения), где есть число или имя (`rewards.slide_facts`), и слов на содержательном слайде.

In [ ]:
from presentation_rl.dsl import parse
from presentation_rl.rewards import Lemmatizer, SentenceEmbedder, slide_facts, slide_texts

embedder, lemmatizer = SentenceEmbedder("BAAI/bge-m3", device="cuda:1"), Lemmatizer()


def slide_diagnostics(answer):
    texts = slide_texts(parse(answer))
    content = texts[1:-1] if len(texts) > 3 else texts[1:]
    content = [t for t in content if t.content.strip()] or content
    if not content:
        return {"facts_share": None, "slide_words": None}
    return {"facts_share": statistics.mean(bool(slide_facts(t.content, lemmatizer)) for t in content),
            "slide_words": statistics.mean(len(t.content.split()) for t in content)}


rows = []
for name in CONFIGS:
    part = evaluate_folder(OUT / name, name, tasks, points, embedder, lemmatizer, build_pptx=False)
    for r in part:
        if not r.get("error"):
            stem = f"{r['mode']}_{r['id']}_{r['sample']}"
            r.update(slide_diagnostics((OUT / name / f"{stem}_answer.txt").read_text(encoding="utf-8")))
    print(f"{name}: эпизодов {len(part)}, ошибок {sum(bool(r.get('error')) for r in part)}")
    rows += part
write_csv(OUT / "episodes.csv", rows)

## 4. Сравнение и правило (до прогона)

Разница `specific − old` по темам (среднее 4 ответов темы), 95% интервал бутстрепа по темам; «лучше» и «хуже» —
интервал целиком по одну сторону от нуля. Промпт с правилом принимается, если (1) хотя бы в одном режиме лучше
покрытие или ключи; (2) ни в одном режиме не хуже покрытие, ключи, общая награда, доля корректных строк и доля
законченных ответов; (3) `#SOURCE` не из выдачи в режиме с поиском не чаще; (4) вырожденных — не больше, чем у `old`,
плюс 5 п.п. Остальное — диагностика.

In [ ]:
METRICS = ["reward_total", "reward_coverage", "reward_keys", "reward_length", "reward_visual", "reward_format",
           "parse_rate", "stop", "slides", "in_range", "conclusion", "searched", "calls", "sources_ok", "sources_bad",
           "generated_tokens", "degenerate", "facts_share", "slide_words"]
table = [c for mode in MODES for metric in METRICS if (c := paired(rows, metric, mode, "old", "specific"))]
write_csv(OUT / "comparison.csv", table)
print(f"{'режим':7s} {'метрика':18s} {'old':>9s} {'specific':>9s} {'разница':>9s}  95% интервал")
for c in table:
    print(f"{c['mode']:7s} {c['metric']:18s} {c['base']:9.3f} {c['other']:9.3f} "
          f"{c['diff']:+9.3f}  [{c['low']:+.3f}, {c['high']:+.3f}]")

In [ ]:
test = lambda metric, mode: paired(rows, metric, mode, "old", "specific")
better = lambda t: t is not None and t["low"] > 0
worse = lambda t: t is not None and t["high"] < 0


def share(name, key="degenerate"):
    part = [r for r in rows if r["model"] == name and not r.get("error")]
    return statistics.mean(bool(r[key]) for r in part)


bad_sources = test("sources_bad", "search")
checks = {
    "gain_coverage_or_keys": any(better(test(m, mode)) for m in ("reward_coverage", "reward_keys") for mode in MODES),
    "no_loss": not any(worse(test(m, mode)) for m in ("reward_coverage", "reward_keys", "reward_total", "parse_rate", "stop")
                       for mode in MODES),
    "sources_bad": not (bad_sources is not None and bad_sources["low"] > 0),
    "degeneration": share("specific") <= share("old") + 0.05,
}
print(checks, f"| вырожденных: old {share('old'):.3f}, specific {share('specific'):.3f}")
ACCEPTED = all(checks.values())
(OUT / "choice.json").write_text(json.dumps({"accepted": ACCEPTED, **checks}, indent=1), encoding="utf-8")
print("ИТОГ: промпт с правилом", "принят" if ACCEPTED else "не принят")

## 5. Пары ответов для чтения

Один и тот же сид, тема и режим: заголовки и текст содержательных слайдов. Файлы целиком — в `OUT/old`
и `OUT/specific`.

In [ ]:
random.seed(0)
PAIRS = random.sample([(mode, key) for mode in MODES for key in IDS], 4)
for mode, key in PAIRS:
    print(f"==================== {mode} {key}: {tasks[key].request.text[:100]}")
    for name in CONFIGS:
        path = OUT / name / f"{mode}_{key}_0_answer.txt"
        if not path.exists():
            continue
        texts = slide_texts(parse(path.read_text(encoding="utf-8")))
        print(f"--- {name}: слайдов {len(texts)}")
        for t in texts[1:5]:
            print(f"  [{t.title}] {t.content[:300]}")